In [ ]:
using BifurcationKit, Plots, Parameters, NLsolve
const BK = BifurcationKit;

# Defining constants and system equations

In [ ]:
const I0 = 1e-12 # A
const κ = 0.7
const Vdd = 1.8 # V
const UT= 25*1e-3 # V
const C=1e-3 # F
;

The sigmoid circuit has (excluding the output branch) 5 transistors for which we can write the full subthreshold transistor equations

\begin{align*}
    \text{M1: }& I_{cmp} = I_{0} e^{\kappa\left(V_{d d}-V_{in}\right) / U_T}\left(1-e^{-\left(V_{d d}-V_1\right) / U_T}\right)\\
    \text{M2: }& I_{1} = I_{0} e^{\kappa V_{thr} / U_T}\left(1-e^{-V_1 / U_T}\right)\\
    \text{M3: }& I_{lin1} = I_{0} e^{\left(\kappa V_1 - V_{out} )\right/ U_T}\left(1-e^{-\left(V_1-V_{out}\right) / U_T}\right)\\
    \text{M4: }& I_{lin2} = I_{0} e^{\left(\kappa V_{out} - V_{2} \right)/ U_T}\left(1-e^{-\left(V_{out}-V_{2}\right) / U_T}\right)\\
    \text{M5: }& I_{lin3} = I_{0} e^{\kappa V_{lin} / U_T}\left(1-e^{-V_2 / U_T}\right)\\
\end{align*}

There are 3 unknown voltages. At each node, we assume there is a (parasitic) capacitor to turn this static system into a dynamic one. For numerical stability we won't be able to take $C$ to be too small. The reason is that simulating such equations can lead to extremely stiff numerical behaviors (because of the various $e^{V/U_T}$ terms, with $U_T=25mV$, so very small with respect to voltage variations ($\approx 1V$))

\begin{align*}
    \text{K1: }& C_1\dot V_1 = I_{cmp} - I_1 - I_{lin1} \\
    \text{Kout: }& C_{out} \dot V_{out} = I_{lin1} - I_{lin2} \\
    \text{K2: }& C_2\dot V_2 = I_{lin2} - I_{lin3}
\end{align*}

In [ ]:
# Static transistor equations
Icmp(Vin,V1) = I0 * exp(κ*(Vdd-Vin)/UT) * (1 - exp(-(Vdd-V1)/UT))
I1(Vthr,V1) = I0 * exp(κ*Vthr/UT) * (1 - exp(-V1/UT))
Ilin1(V1,Vout) = I0 * exp((κ*V1 - Vout)/UT) * (1 - exp(-(V1-Vout)/UT))
Ilin2(Vout,V2) = I0 * exp((κ*Vout - V2)/UT) * (1 - exp(-(Vout-V2)/UT))
Ilin3(Vlin,V2) = I0 * exp(κ*Vlin/UT) * (1 - exp(-V2/UT));

In [ ]:
function Imode_sigmoid_V(x,pars)

	@unpack Vin, Vlin, Vthr = pars

    V1, Vout, V2 = x

    [Icmp(Vin,x[1]) - I1(Vthr,x[1]) - Ilin1(x[1],x[2]),
    Ilin1(x[1],x[2]) - Ilin2(x[2],x[3]),
    Ilin2(x[2],x[3]) - Ilin3(Vlin,x[3])] ./C
    
end;

In order to set the various control voltages in suitable ranges (close to those used in the actual circuit), we can use current generators and diode-connected transistors to map control currents to control voltages:

$$V_{P_\text{diode}} = V_{dd} - \frac{U_T}{\kappa} \log \left(\frac{I_{in}}{I_0}\right)$$
$$V_{N_\text{diode}} = \frac{U_T}{\kappa} \log \left(\frac{I_{in}}{I_0}\right)$$

In [ ]:
V_P_diode(I) = Vdd - UT/κ * log(I/I0)
V_N_diode(I) = UT/κ * log(I/I0);

After simulating the system, we will convert the input and output voltages to current using these equations:

\begin{align}
    I_{in} &= I_0 e^{\kappa (V_{dd}-V_{in})/U_T}\\
    I_{out} &= I_0 \frac{e^{\kappa V_{out}/U_T}}{1+e^{\kappa\left(V_{out} - V_{gain}\right)/U_T}}
\end{align}

In [ ]:
Iin(Vin) = I0 * exp(κ*(Vdd-Vin)/UT)
Iout(Vout) = I0 * exp((κ*Vout)/UT) / (1 + exp(κ*(Vout-Vgain)/UT));

# Solving the equations
## Parameter definition

- $I_{thr}$: Sets the lower threshold (the zero point) of the sigmoid
- $I_{gain}$: Sets the gain of the sigmoid (its maximum output current)
- $I_{lin}$: Sets the linear range of the sigmoid (should be in the same range as $I_{gain}$)
- $I_{in}$: Input current

In [ ]:
Ithr = 100e-9 # A
Igain = 500e-9 # A
Ilin = 300e-9 # A

Iin_range = (1e-9, 600e-9);

## Guess of initial point on the sigmoid characteristic

We can use Julia NLsolve to obtain "good" initial conditions to simulate the system and get a first point of the sigmoid input-output characteristic.

In [ ]:
# Current mirror parameter conversion
Vthr = V_N_diode(Ithr)
Vgain = V_N_diode(Igain)
Vlin = V_N_diode(Ilin)
Vin = V_P_diode(Igain) # We want to make an initial guess with a high current to avoid very small values for our variables

pars_V = (Vin = Vin, Vlin = Vlin, Vthr = Vthr)

x0=[1.7, 0.9, 0.3]

solNL = nlsolve(x -> Imode_sigmoid_V(x,pars_V), x0, iterations=convert(Int64,1e6), ftol=1e-9, xtol=1e-6)

## Circuit simulation
We use Julia BifurcationKit to compute the steady-states of our system to retrieve the sigmoid characteristic

In [ ]:
rfs(x, p) = (x2 = x[2], y=p) # Record the output voltage
prob = BifurcationProblem(Imode_sigmoid_V, solNL.zero, pars_V, (@lens _.Vin), record_from_solution = rfs) # Set up the bifurcation problem with the input voltage as the bifurcation parameter

In [ ]:
# We set up continuation to use the selected input current range, by converting them to PMOS current mirror voltages
opts = ContinuationPar(p_min = V_P_diode(Iin_range[2]), p_max = V_P_diode(Iin_range[1]), n_inversion = 50, ds = 1e-6, dsmin = 1e-12, dsmax = 1e-3, max_steps = 1000, nev = 3)
br = continuation(prob, PALC(), opts; normC = norminf, bothside = true)

In [ ]:
# If everything went well, we should only obtain one branch which we can obtain directly without the need of a full bifurcation diagram
plot(Iin.(br.branch.param), Iout.(br.branch.x2), grid=false, xlabel="Iin", ylabel="Iout", legend=false,title="Ithr=$Ithr, Igain=$Igain, Ilin=$Ilin")